In [1]:
import requests

import pandas as pd

from datasets import load_dataset

In [72]:
ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1066 entries, 0 to 1065
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    1066 non-null   object
 1   label   1066 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 16.8+ KB


In [73]:
test['label'] = test['label'].apply(lambda x: 'positive' if x == 1 else 'negative')

labels = test['label'].unique()

test

,text,label
0,lovingly photographed in the manner of a golde...,positive
1,consistently clever and suspenseful .,positive
2,"it's like a "" big chill "" reunion of the baade...",positive
3,the story gives ample opportunity for large-sc...,positive
4,"red dragon "" never cuts corners .",positive
...,...,...
1061,a terrible movie that some people will neverth...,negative
1062,there are many definitions of 'time waster' bu...,negative
1063,"as it stands , crocodile hunter has the hurrie...",negative
1064,the thing looks like a made-for-home-video qui...,negative


In [74]:
test = test.sample(1)

In [ ]:
def generate_text(messages, url):
    payload = {
        "model": "deepseek-r1:1.5b",
        "messages" : messages,
        "stream": False,
        "options": {
            "temperature": 0.5,
            "num_predict": 5000,
        }
    }

    response = requests.post(url, json=payload)

    return response.json()

messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": f"who is the president of the USA?"
    },
    {
        "role": "assistant",
        "content": "<think> Okay, so the last time I checked, the president of the US"
    }
]

url = "http://localhost:11434/api/chat"

response = generate_text(messages, url)

In [6]:
print(response['message'])

{'role': 'assistant', 'content': ' was someone else. Now I\'m trying to figure out who it is. Let me think about what I know.\n\nI remember that the president is usually called the "president" or "president of the United States." They hold the office for a term before they\'re elected. Oh right, there\'s also something called the vice president, which I believe takes over after the president in some cases.\n\nWait, but how long does each term last? I think it was three terms when I was younger, but now it\'s probably longer. Maybe five or seven. That must have been a lot for the president and the vice president during their time as president.\n\nI remember hearing about someone named Bill. Is he the current president? Or is it someone else like Joe Biden? Let me try to recall. There was a time when I saw something about him, maybe in a documentary or a movie. But now I\'m not sure if he\'s the current one.\n\nI think the current president is someone named Joe Biden. He took office afte

In [2]:
def generate_text2(message, url):
    payload = {
        "model": "deepseek-r1:1.5b",
        "prompt": message,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "num_predict": 500,
        },
        "raw": True
    }

    response = requests.post(url, json=payload)
    response = response.json()

    message = response['response'].strip().lower()

    return message, response

message = f"""[INST]
System: You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling binary classification tasks based on user instructions.
User: Classify the following text based on the task: Sentiment analysis of possibly hateful content. Only respond with the label that best describe the text. The possible labels are: hateful, not hateful. Tweet:
[/INST]
"""
url = "http://localhost:11434/api/generate"

response = generate_text2(message, url)
response

('text:\n\nthe text is about a person who has been accused of hate speech and is being evaluated for whether they have any hate-related issues.\nit mentions specific instances where the accused person used words like "white," "black," and "hispanic."\nadditionally, it states that the accused person has no history of hate-related issues in their past.\nthe text does not provide any information about the accused person\'s background or other relevant details.\n\nbased on this analysis, what is the correct label for the text? the options are hateful or not hateful. to determine the label, i need to assess whether the text supports the claim that it contains hate-related content. if the text discusses specific instances of hate speech and provides evidence that the accused person has no history of such issues, then it would be classified as "not hateful." however, if the text does not provide any information about hate speech or lacks relevant context, it would be labeled as "not hateful."

In [15]:
response.split('</think>')[1]

'\n\nhate'

In [141]:
def classify(text, labels):
    url = "http://localhost:11434/api/chat"

    messages = [
            {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling binary classification tasks based on user instructions."},
            {"role": "user", "content": f"Classify the following text based on the task: Sentiment analysis of movie reviews. Only respond with the label that best describe the text The possible labels are: {', '.join(labels)}. Text: {text}"}
        ]

    print(messages)
    
    response = generate_text(messages, url)
    
    message = response['message']
    
    if '</think>' not in message['content']:
        print('Adding </think>')
        message['content'] += '</think>'

        messages.append(message)

        response = generate_text(messages, url)

        content = response['message']['content'].lower()

    else:
        content = response['message']['content'].lower()
        content = content.split('</think>')[1]

    print(content)

    return content.strip()



test['prediction'] = test['text'].apply(lambda x: classify(x, labels)).apply(pd.Series)

[{'role': 'system', 'content': 'You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling binary classification tasks based on user instructions.'}, {'role': 'user', 'content': 'Classify the following text based on the task: Sentiment analysis of movie reviews. Only respond with the label that best describe the text The possible labels are: positive, negative. Text: a dim-witted and lazy spin-off of the animal planet documentary series , crocodile hunter is entertainment opportunism at its most glaring .'}]
<Response [200]>


negative


In [133]:
# print (test['prediction'])
print(test['prediction'].value_counts())

prediction
negative    1
Name: count, dtype: int64


In [57]:
url = "http://localhost:11434/api/chat"

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": f"Who is the tallest human?"}
]

response = generate_text(messages, url)

message = response['message']
if '</think>' not in message['content']:
    message['content'] += '</think>'

    

In [ ]:
messages.append(message)
messages

In [62]:
response = generate_text(messages, url)
response

{'model': 'deepseek-r1:1.5b',
 'created_at': '2025-03-29T19:23:29.2540765Z',
 'message': {'role': 'assistant',
  'content': "\n\nThe tallest human is a subject of debate among historians and scientists. While there are records of individuals reaching significant heights, the exact tallest person isn't definitively known due to various factors like genetic diversity and measurement inaccuracies. However, estimates suggest that some individuals can reach up to 6 feet or more, but precise measurements remain uncertain.\n\nAnswer: The tallest human is not definitively known as a single individual, but records indicate that some people can be significantly taller than average. While there are no exact records of the tallest person, estimates suggest that some individuals may reach around 6 feet or more."},
 'done_reason': 'stop',
 'done': True,
 'total_duration': 1660893000,
 'load_duration': 17928900,
 'prompt_eval_count': 216,
 'prompt_eval_duration': 3131000,
 'eval_count': 122,
 'eval_d

In [54]:
response

{'model': 'deepseek-r1:1.5b',
 'created_at': '2025-03-29T19:02:42.2216154Z',
 'message': {'role': 'assistant',
  'content': '<think>\nOkay, so I\'m trying to figure out who is the tallest human. Hmm, that\'s an interesting question. I know that height varies among people, but I\'m not sure how tall someone can be. Let me think about this step by step.\n\nFirst, I should consider what defines a "tallest" person. It\'s probably based on height measurements, so maybe using some kind of standardized unit like meters or feet. But wait, different countries and regions might have slightly different standards for measuring height. That could affect the answer.\n\nI remember hearing about records where people were measured to see how tall they could be. For example, in the United States, there\'s a record for standing up to 6\'1" (73 inches) without bending over. But I\'m not sure if that\'s the tallest anyone has ever been. Maybe it\'s part of a larger record.\n\nI also think about the concept

In [48]:
content = response.json()
content = content['message']
content['content'] += '</think>'
messages.append(content)

payload = {
    "model": "deepseek-r1:1.5b",
    "messages" : messages,
    "stream": False,
    "options": {
        "temperature": 0.0,
        "num_predict": 200,
    }
}

response = requests.post(url, json=payload)
response.json()

{'model': 'deepseek-r1:1.5b',
 'created_at': '2025-03-29T18:28:20.035736Z',
 'message': {'role': 'assistant',
  'content': '\n\nThe tallest human records are quite impressive, but they depend on the specific measurements and standards used. For example, in the United States, there have been records where people stand up to 6\'1" (73 inches) without bending over. However, these records might not be the absolute tallest anyone has ever achieved.\n\nIn terms of height as perceived from a distance, some individuals can appear very tall due to perspective changes. For instance, someone who is 5\'6" (66 inches) standing at arm\'s length could seem much taller than they actually are when viewed from a closer point.\n\nIt\'s important to note that the tallest person ever recorded on Earth was about 7 feet 1 inch (83 inches) tall, achieved by Usain Bolt in 2009. This record is based on standing height without bending over.\n\nSo, while there have been records of very tall individuals, the absol